# Day 87 — Training

This notebook covers End-to-End PyTorch Mini-Batch Training Loop (`train_one_epoch`, `fit_model`), Gradient Zeroing (`optimizer.zero_grad()`), Regression Loss (`nn.MSELoss`), Adaptive Optimization (`torch.optim.Adam` with `weight_decay`), and Sample-Weighted Metric Aggregation (`MSE`, `RMSE`, `MAE`)

## Validation goals
- Implement the five-step mini-batch training sequence (device transfer, `optimizer.zero_grad()`, forward pass, `nn.MSELoss()` computation, `loss.backward()`, and `optimizer.step()`)
- Build modular `train_one_epoch` and `fit_model` functions to train `RetailDemandMLP` across 25 epochs on the 12-feature retail demand dataset
- Compute sample-weighted epoch metrics (`MSE`, `RMSE`, and `MAE` in `demand_units`) to prevent partial-batch averaging bias when `drop_last=False`
- Analyze why `optimizer.zero_grad()` precedes `loss.backward()`, how `nn.MSELoss()` complements `MAE`, and how `weight_decay=1e-4` applies L2 regularization

### 1. Anatomy of a PyTorch Training Step
Each mini-batch inside an epoch executes five core steps in sequence:
- Move Tensors to Device: Send `batch_x` and `batch_y` to device (cpu or cuda) so they match the model's parameters.
- Zero Old Gradients (`optimizer.zero_grad()`): Clear gradients accumulated during the previous step so old slopes do not bleed into the current update.
- Forward Pass (`preds = model(batch_x)`): Compute predicted `demand_units` for the mini-batch.
- Compute Loss (`loss = criterion(preds, batch_y)`): Measure error against `batch_y` using a regression loss such as `nn.MSELoss()` or `nn.SmoothL1Loss()` (Huber loss).
- Backward Pass & Parameter Update (`loss.backward()` then `optimizer.step()`): Backpropagate gradients through the graph and update weights and biases.

### 2. Implementing the Training Loop (`day_87_training.py`)
For continuous unit forecasting where occasional demand spikes or outliers can occur, `nn.MSELoss()` penalizes larger errors quadratically, while optim.Adam adapts per-parameter step sizes smoothly across features of different scales. Tracking both MSE and MAE (in `demand_units`) gives you both optimization smoothness and intuitive unit error readouts.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from day_85_dataset import build_dataloaders
from day_86_architecture import RetailDemandMLP


def set_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """Runs one full pass over training batches and returns (mean_mse, mean_mae)."""
    model.train()
    total_mse_sum = 0.0
    total_mae_sum = 0.0
    total_samples = 0

    for batch_x, batch_y in dataloader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_size = batch_x.shape[0]

        optimizer.zero_grad()
        preds = model(batch_x)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            mae_batch = torch.abs(preds - batch_y).sum().item()
            total_mse_sum += loss.item() * batch_size
            total_mae_sum += mae_batch
            total_samples += batch_size

    return total_mse_sum / total_samples, total_mae_sum / total_samples


def fit_model(
    model: nn.Module,
    train_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    epochs: int = 25,
    log_every: int = 5,
) -> list[dict[str, float]]:
    """Trains the model across multiple epochs and records training history."""
    history: list[dict[str, float]] = []

    for epoch in range(1, epochs + 1):
        train_mse, train_mae = train_one_epoch(
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
        )
        train_rmse = float(np.sqrt(train_mse))
        record = {
            "epoch": float(epoch),
            "train_mse": train_mse,
            "train_rmse": train_rmse,
            "train_mae": train_mae,
        }
        history.append(record)

        if epoch == 1 or epoch % log_every == 0 or epoch == epochs:
            print(
                f"Epoch {epoch:2d}/{epochs:2d} | "
                f"Train MSE: {train_mse:8.2f} | "
                f"RMSE: {train_rmse:6.2f} units | "
                f"MAE: {train_mae:6.2f} units"
            )

    return history


if __name__ == "__main__":
    set_seed(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, _, num_features, source = build_dataloaders(batch_size=32)
    model = RetailDemandMLP(
        in_features=num_features,
        hidden_dims=(64, 32),
        dropout=0.1,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)

    print("--- Day 87: Neural Network Training Loop ---")
    print(f"Data source: {source} | Device: {device} | Features: {num_features}\n")

    history = fit_model(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=25,
        log_every=5,
    )

--- Day 87: Neural Network Training Loop ---
Data source: synthetic_fallback | Device: cpu | Features: 12

Epoch  1/25 | Train MSE: 22829.25 | RMSE: 151.09 units | MAE: 146.66 units
Epoch  5/25 | Train MSE:   473.22 | RMSE:  21.75 units | MAE:  17.08 units
Epoch 10/25 | Train MSE:   438.63 | RMSE:  20.94 units | MAE:  16.84 units
Epoch 15/25 | Train MSE:   469.34 | RMSE:  21.66 units | MAE:  17.22 units
Epoch 20/25 | Train MSE:   442.73 | RMSE:  21.04 units | MAE:  16.52 units
Epoch 25/25 | Train MSE:   453.42 | RMSE:  21.29 units | MAE:  16.58 units


### 3. Practical Engineering Details in the Loop

- Sample-Weighted Averaging Across Batches: Because the last batch in a DataLoader may be smaller than `batch_size`, multiplying each batch's mean loss by `batch_size` and dividing by total_samples gives the true dataset-wide average rather than an unweighted average of uneven batches.
- Weight Decay (`weight_decay=1e-4`): Adds L2 regularization directly inside Adam, gently pulling large weights toward zero to help prevent overfitting.
- Why `model.train()` Is Explicit: Calling `model.train()` at the top of train_one_epoch guarantees the network is in training mode even if an evaluation helper called `model.eval()` earlier.

### Guided Question:

Why must `optimizer.zero_grad()` run before `loss.backward()` rather than after `optimizer.step()`?

### Answer:

In PyTorch, `loss.backward()` accumulates (+=) new gradients into param.grad rather than overwriting them. Calling `optimizer.zero_grad()` before `loss.backward()` clears stale gradients from the previous step so the new gradients belong strictly to the current mini-batch.

**Why Before `loss.backward()` Matters**

- Prevents accidental gradient accumulation: If you run `loss.backward()` without zeroing first, gradients from step t - 1 stay in .grad and get added to step t. On standard mini-batch training, that mixes gradients from different batches and distorts parameter updates.
- Keeps the loop clean and safe: Placing `optimizer.zero_grad()` right before `loss.backward()` ensures .grad starts fresh every time you compute gradients for a batch.

**What Happens If You Put It After `optimizer.step()`**

- First step still starts dirty: Before the very first training step (or if .grad was populated earlier, such as during a manual inspection or pre-step check), old gradients may already exist.
- Breaks multi-step accumulation: When you intentionally accumulate gradients across multiple mini-batches (for example, stepping only every k batches), clearing right after `optimizer.step()` works only if you track accumulation counts carefully. Putting `optimizer.zero_grad()` right before `loss.backward()` makes the intent explicit: "clear before computing new gradients."

### Guided Question:

Why is `nn.MSELoss()` useful as the training objective while MAE (in `demand_units`) is helpful to read alongside it?

### Answer:

Using `nn.MSELoss()` as the training objective gives smooth, larger gradients for large demand errors during optimization, while tracking MAE (Mean Absolute Error) in demand_units gives an intuitive, scale-matched measure of average forecast error in real units.

**Why `nn.MSELoss()` Helps During Training**

- Smooth, differentiable optimization: Squared error $(y - \hat{y})^2$ is smooth everywhere and produces gradients proportional to the error ($2(\hat{y} - y)$). Large mistakes generate stronger corrective pulls than small ones, helping the model converge smoothly.
- Penalizes big misses heavily: Because errors are squared, a forecast that is off by 40 units gets 16 times the penalty of a forecast off by 10 units—useful when large demand surges or stockouts are especially costly.
- Clean alignment with conditional means: Minimizing squared error naturally pushes predictions toward the conditional mean of demand given the features.

**Why MAE in demand_units Helps During Evaluation**

- Directly interpretable in business units: `MSE` is in squared demand units (`demand_units^2`), which is hard to picture intuitively. MAE stays in demand_units—an MAE of 12 means predictions are off by about 12 units on average.
- Less dominated by extreme spikes: Because MAE weights every unit of error linearly ($\vert{}y - \hat{y}\vert{}$) rather than squaring it, a single extreme promotion or outlier day does not inflate the metric as dramatically as `MSE` or `RMSE`.
- Complements the training loss: Watching `MSELoss` tells you how well optimization and large-error penalties are behaving, while watching `MAE` tells you the typical day-to-day unit error stakeholders actually care about.

### Guided Question:

When `drop_last=False` produces a smaller final batch, why is weighting batch loss by batch_size before dividing by total_samples more accurate than averaging per-batch losses directly?

### Answer:

When `drop_last=False` leaves a smaller final batch, standard per-batch losses (like `nn.MSELoss()` or `nn.L1Loss()`) return a mean over that batch, not a sum. Weighting each batch's mean loss by its batch_size before dividing by total_samples recovers the true sample-weighted average across the entire dataset instead of giving every batch equal vote.

**Why Direct Batch Averaging Is Biased**

Suppose a dataset has $100$ samples and `batch_size=32`, producing three batches of size $32$ and a final batch of size $4$:
- Direct batch mean: $\frac{1}{4}(L_1 + L_2 + L_3 + L_4)$ weights the $4$-sample batch at $25\%$ of the total score—even though it represents only $4\%$ of the data.
- Sample-weighted mean: $\frac{32 L_1 + 32 L_2 + 32 L_3 + 4 L_4}{100}$ weights each batch by its actual share of samples ($\frac{32}{100}$, $\frac{32}{100}$, $\frac{32}{100}$, and $\frac{4}{100}$).

**Key Benefits of Sample Weighting**

- Matches full-dataset evaluation: Because every individual sample contributes $\frac{1}{N}$ to the final metric, the weighted batch average equals what you would get computing the loss over all `total_samples` at once.
- Prevents small-batch distortion: If the final partial batch happens to have unusually high or low error (or contains only a few rows), direct batch averaging distorts epoch loss much more noticeably—especially on smaller datasets or larger batch sizes.
- Stabilizes comparisons across batch sizes: Changing `batch_size` changes the size of the remainder batch; weighting by `batch_size` keeps epoch-level loss consistent and comparable across runs.